# 01 — S7 pilot

Tests of the forms and the probe pipeline on the development set. **Not results.** Requires the local corpus (`scripts/fetch_core.py`).

In [1]:
import collections
import csv
import json
from pathlib import Path

ROOT = Path('..')
rows = list(csv.DictReader(open(ROOT/'data/copy_seeds.csv')))
collections.Counter((r['earlier'], r['later'], r['guess']) for r in rows).most_common(12)

[(('31', '21', 'mentioned, relation unclear'), 84),
 (('31', '34', 'mentioned, relation unclear'), 81),
 (('34', '51', 'mentioned, relation unclear'), 65),
 (('51', '21', 'mentioned, relation unclear'), 63),
 (('34', '51', 'free copy'), 18),
 (('51', '21', 'free copy'), 16),
 (('31', '34', 'reversed copy'), 11),
 (('51', '21', 'reversed copy'), 10),
 (('51', '21', 'not mentioned'), 10),
 (('31', '21', 'not mentioned'), 9),
 (('34', '51', 'not mentioned'), 9),
 (('31', '34', 'free copy'), 7)]

## Blind inventories (Stage A form test)

In [2]:
inv = {p.stem: [l.split('\t') for l in p.read_text().splitlines() if l.strip()] for p in (ROOT/'annotations/pilot/A_inventory').glob('p*.tsv')}
conf = collections.Counter(l[2].strip() for v in inv.values() for l in v if len(l) == 3)
types = collections.Counter(l[0] for v in inv.values() for l in v)
len(inv), conf, types

(79,
 Counter({'clear': 602, 'probable': 197, 'indeterminate': 56}),
 Counter({'ENTITY': 340,
          'ACTION': 134,
          'ATTRIBUTE': 127,
          'RELATION': 88,
          'SETTING': 79,
          'RECOGNISED': 78,
          'INSCRIPTION': 9}))

## Probe pipeline

In [3]:
r = [json.loads(l) for l in open(ROOT/'results/probes/qwen2.5vl_3b/pilot_probes.jsonl')]
for c in ['image','image+motto','image+epigram','image+mismatched','text_only','mirrored+epigram']:
    rs = [x for x in r if x['condition'] == c]
    print(f"{c:18s} picture={sum(x['answer']==x['picture_answer'] for x in rs):2d} text={sum(x['answer']==x['text_answer'] for x in rs):2d} / {len(rs)}")
collections.Counter(x['answer'] for x in r)

image              picture=17 text= 7 / 24
image+motto        picture=16 text= 8 / 24
image+epigram      picture=10 text=14 / 24
image+mismatched   picture=19 text= 5 / 24
text_only          picture= 9 text=15 / 24
mirrored+epigram   picture=11 text=13 / 24


Counter({'yes': 88, 'no': 56})